# <u>EXTRA - Modelos de Pronósticos de Series de Tiempo</u>

## Introduction:

El conjunto de datos proporciona las ventas mensuales de champán desde enero de 1964 hasta septiembre de 1972 para la marca Perrin Freres. Proporciona el número de ventas mensuales de champán desde enero de 1964 hasta septiembre de 1972. Los valores son un recuento de millones de ventas y hay 105 observaciones. El conjunto de datos se atribuye a Makridakis y Wheelwright, 1989.


## <a id='1.'>1. Importing Libraries</a>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math
import random

import xgboost as xgb
from xgboost import plot_importance, plot_tree


from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.style.use('fivethirtyeight')

In [ ]:
import os
# Let cmdstanpy know where CmdStan is
os.environ["CMDSTAN"] = "./cmdstan-2.23.0"

from prophet import Prophet

## <a id='2.'>2. Understanding Data</a>

In [ ]:
# from google.colab import files
# uploaded = files.upload()

In [ ]:
# Loading the data
import io
import pandas as pd
df = pd.read_csv('champagne.csv',encoding='latin-1', sep = ',')
df.head()

In [ ]:
df.describe()

In [ ]:
# Plot the data
plt.figure(figsize=(15,6))
plt.plot(df.iloc[:,1].values)
plt.xlabel("time")
plt.ylabel("Number of Sales")
plt.title("Champagne Sales")
plt.show()

In [ ]:
df.info()

## <a id='3.'>3. Implementing FBProphet</a>

Prophet, o "Facebook Prophet", es una biblioteca de código abierto para el pronóstico de series de tiempo univariante (una variable) desarrollada por Facebook. Prophet implementa lo que ellos denominan un modelo de pronóstico de series de tiempo aditivo, y la implementación admite tendencias, estacionalidad y días festivos.


## <a id='3.1.'>3.1. Preprocessing for FBProphet</a>

In [ ]:
# Check the end of the dataset.
df.tail(12)

In [ ]:
# Train-Test-Split
train_df = df.loc[:93].copy()
test_df = df.loc[93:].copy()

In [ ]:
#Checking the shapes
print(train_df.shape)
print(test_df.shape)

In [ ]:
# Plotting train and test split
train_df.rename(columns={'Sales': 'TRAIN SET'}).merge(test_df.rename(columns={'Sales': 'TEST SET'}),how='outer').plot(figsize=(8,4), title='Sales', style="-")
plt.show()

## <a id='3.2.'>3.2. FBProphet Model</a>
El modelo Prophet espera que el conjunto de datos se denomine de una manera específica. Cambiaremos el nombre de nuestras columnas de marco de datos antes de introducirlas en el modelo.


In [ ]:
# Format data for prophet model using ds and y
train_df = train_df.reset_index().rename(columns={'Month':'ds','Sales':'y'})
train_df.drop(["index"], axis=1, inplace=True)
train_df.head()

In [ ]:
# Format data for prophet model using ds and y
test_df = test_df.reset_index().rename(columns={'Month':'ds','Sales':'y'})
test_df.drop(["index"], axis=1, inplace=True)
test_df.head()

In [ ]:
# Setup and train model and fit
model = Prophet()
model.fit(train_df)

In [ ]:
# Predict on test set with model
forecast = model.predict(test_df)

In [ ]:
forecast.head()

In [ ]:
# Plot the forecast
f, ax = plt.subplots(1)
f.set_figheight(5)
f.set_figwidth(15)
fig = model.plot(forecast,ax=ax)
plt.show()

In [ ]:
# Plot the components of the model
fig = model.plot_components(forecast)

In [ ]:
test_df['Prediction'] = forecast["yhat"]

In [ ]:
test_df

In [ ]:
prophet_plot = pd.concat([train_df,test_df], sort=False)

In [ ]:
prophet_plot.reset_index(inplace=True)

In [ ]:
prophet_plot.tail(13)

In [ ]:
f, ax = plt.subplots(figsize=(14,5))
f.set_figheight(5)
f.set_figwidth(15)
prophet_plot.plot(kind='line',x='ds', y='y', color='red', label='Actual', ax=ax)
prophet_plot.plot(kind='line',x='ds',y='Prediction', color='blue',label='Forecast', ax=ax)
plt.title("Forecast vs Actuals")
plt.show()

In [ ]:
# Plot the forecast with the actuals
f, ax = plt.subplots(1)
f.set_figheight(5)
f.set_figwidth(10)
_ = prophet_plot[['Prediction','y']].plot(ax=ax, style=['-','.'])
ax.set_xbound(lower=93, upper=104)
ax.set_ylim(0, 15000)
plot = plt.suptitle('Forecast vs Actuals')

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error, mean_squared_log_error

In [ ]:
def evaluate_forecast(y,pred):
    results = pd.DataFrame({'r2_score':r2_score(y, pred),
                           }, index=[0])
    results['mean_absolute_error'] = mean_absolute_error(y, pred)
    results['median_absolute_error'] = median_absolute_error(y, pred)
    results['mse'] = mean_squared_error(y, pred)
    results['msle'] = mean_squared_log_error(y, pred)
    results['rmse'] = np.sqrt(results['mse'])
    return results

In [ ]:
y_true=test_df['y']
y_pred=forecast['yhat']

evaluate_forecast(y_true, y_pred)

## <a id='4.'>4. Implementing XGBoost</a>

XGBoost es una biblioteca optimizada de aumento de gradiente distribuida diseñada para ser altamente eficiente, flexible y portátil. Implementa algoritmos de aprendizaje automático en el marco de Gradient Boosting. XGBoost proporciona un aumento de árbol paralelo (también conocido como GBDT, GBM) que resuelve muchos problemas de ciencia de datos de una manera rápida y precisa.


## <a id='4.1.'>4.1. Preprocessing for XGBoost</a>

In [ ]:
df_xgb = df.copy()
df_xgb.head()

XGBoost requires that the time series dataset be transformed into a supervised learning problem first.

In [ ]:
# new data frame with split value columns
new = df_xgb["Month"].str.split("-", n = 1, expand = True)

# making separate first name column from new data frame
df_xgb["year"]= new[0]

# making separate last name column from new data frame
df_xgb["month"]= new[1]

#remane
df_xgb = df_xgb.rename(columns={'Month': 'date'})

#Changing data types
df_xgb['year'] = df_xgb['year'].astype(int)
df_xgb['month'] =df_xgb['month'].astype(int)

df_xgb.drop(["date"], axis=1, inplace=True)

# df display
df_xgb.head()

In [ ]:
df_xgb.info()

In [ ]:
# Train-Test-Split for XGBoost
train_df = df_xgb.loc[:92].copy()
test_df = df_xgb.loc[93:].copy()

In [ ]:
# Preparing train-test data
x_train = train_df.drop(["Sales"], axis=1)
y_train = train_df["Sales"]
x_test = test_df.drop(["Sales"], axis=1)
y_test = test_df["Sales"]

In [ ]:
# Checking the shapes
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

## <a id='4.2.'>4.2. XGBoost Model</a>

In [ ]:
xgb_model = xgb.XGBRegressor(n_estimators=1000)
xgb_model.fit(x_train, y_train,
        eval_set=[(x_train, y_train), (x_test, y_test)],
       verbose=True)

In [ ]:
_ = plot_importance(xgb_model, height=0.9)

In [ ]:
test_df['Prediction'] = xgb_model.predict(x_test)
xgboost_all = pd.concat([test_df, train_df], sort=False)

In [ ]:
_ = xgboost_all[['Sales','Prediction']].plot(figsize=(15, 5))

In [ ]:
# Plot the forecast with the actuals
f, ax = plt.subplots(1)
f.set_figheight(5)
f.set_figwidth(15)
_ = xgboost_all[['Prediction','Sales']].plot(ax=ax, style=['-','.'])
ax.set_xbound(lower=93, upper=105)
ax.set_ylim(0, 15000)
plot = plt.suptitle('Forecast vs Actuals')

In [ ]:
y_true=test_df['Sales']
y_pred=test_df['Prediction']

evaluate_forecast(y_true, y_pred)

## <a id='5.'>5. Implementing LSTM </a>

## <a id='5.1.'>5.1. Preprocessing for LSTM </a>

In [ ]:
df.head()

In [ ]:
df.Month = pd.to_datetime(df.Month)

In [ ]:
df = df.set_index("Month")
df.head()

In [ ]:
df.index.freq = 'MS'

In [ ]:
train_data = df[:len(df)-12]
test_data = df[len(df)-12:]

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

In [ ]:
scaler.fit(train_data)
scaled_train_data = scaler.transform(train_data)
scaled_test_data = scaler.transform(test_data)

## <a id='5.2.'>5.2. LSTM Model </a>

In [ ]:
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

n_input = 12 #Month
n_features= 1 #Sales
generator = TimeseriesGenerator(scaled_train_data, scaled_train_data, length=n_input, batch_size=1)

In [ ]:
model = Sequential()
model.add(LSTM(300, activation='relu', return_sequences=True, input_shape=(n_input, n_features)))
model.add(LSTM(300, activation='relu'))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
model.fit(generator,epochs=10)

In [ ]:
# Checking the loss
losses_lstm = model.history.history['loss']
plt.figure(figsize=(12,4))
plt.xticks(np.arange(0,21,1))
plt.plot(range(len(losses_lstm)),losses_lstm);

In [ ]:
lstm_predictions_scaled = list()

batch = scaled_train_data[-n_input:]
current_batch = batch.reshape((1, n_input, n_features))

for i in range(len(test_data)):
    lstm_pred = model.predict(current_batch)[0]
    lstm_predictions_scaled.append(lstm_pred)
    current_batch = np.append(current_batch[:,1:,:],[[lstm_pred]],axis=1)

In [ ]:
lstm_predictions_scaled

In [ ]:
lstm_predictions = scaler.inverse_transform(lstm_predictions_scaled)

In [ ]:
lstm_predictions

In [ ]:
test_data['LSTM_Predictions'] = lstm_predictions

In [ ]:
test_data

In [ ]:
test_data['Sales'].plot(figsize = (16,5), legend=True)
test_data['LSTM_Predictions'].plot(legend = True);

In [ ]:
y_true=test_data['Sales']
y_pred=test_data['LSTM_Predictions']

evaluate_forecast(y_true, y_pred)